In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
file_path = r'C:\dev\projects\NLP_test\data\train_dataset_manual.csv'
data = pd.read_csv(file_path)
data.describe()

,max(page),target,available
count,150,54,150
unique,150,52,2
top,https://www.factorybuys.com.au/products/euro-t...,Gift Card,True
freq,1,3,78


In [3]:
data['target'] = data['target'].fillna('NO_PRODUCT')

In [4]:
data.head(10)

,max(page),target,available
0,https://www.factorybuys.com.au/products/euro-t...,Factory Buys 32cm Euro Top Mattress - King,True
1,https://dunlin.com.au/products/beadlight-cirrus,Beadlight Cirrus LED Reading Light,True
2,https://themodern.net.au/products/hamar-plant-...,Hamar Plant Stand - Ash,True
3,https://furniturefetish.com.au/products/oslo-o...,NO_PRODUCT,False
4,https://hemisphereliving.com.au/products/,NO_PRODUCT,False
5,https://home-buy.com.au/products/bridger-penda...,NO_PRODUCT,False
6,https://interiorsonline.com.au/products/interi...,"$50 RJ Living Gift Card, $100 RJ Living Gift C...",True
7,https://beckurbanfurniture.com.au/products/pag...,NO_PRODUCT,False
8,https://livingedge.com.au/products/tables/dining,"Linear Wood Table, Saarinen Dining Table, Oval...",True
9,https://edenliving.online/collections/summerlo...,NO_PRODUCT,False


In [5]:
train_data = data[data["available"] == True]

In [6]:
train_data.describe()

,max(page),target,available
count,78,78,78
unique,78,52,1
top,https://www.factorybuys.com.au/products/euro-t...,NO_PRODUCT,True
freq,1,25,78


In [7]:
from sklearn.model_selection import train_test_split
X, y = train_data['max(page)'], train_data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [8]:
import re
import json
import random
from math import ceil
from urllib.parse import urljoin, urlparse
from typing import Optional, Callable

import requests
from bs4 import BeautifulSoup, Tag

TIMEOUT = 30

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

_SKIP_TAGS = {"header", "footer", "aside", "nav"}
_SKIP_ATTRS = re.compile(
    r"\b(nav|footer|header|menu|breadcrumb|sidebar|filter|facet|pagination|"
    r"cart|checkout|newsletter|popup|modal|cookie|banner|subscribe|"
    r"social|share|wishlist-modal|refinement)\b",
    re.IGNORECASE,
)

_BAD_TEXT_RE = re.compile(
    r"\b(cookie|privacy|policy|terms|sign in|log in|subscribe|newsletter|"
    r"javascript|wishlist|compare|share|sort by|filter by|view more|load more)\b",
    re.IGNORECASE,
)

_CARD_PATTERN = re.compile(
    r"\b(product[-_]?card|product[-_]?tile|product[-_]?item|"
    r"grid[-_]?item|collection[-_]?item|plp[-_]item|"
    r"ProductCard|productCard)\b",
    re.IGNORECASE,
)

_PRODUCT_URL_PATTERNS = [
    re.compile(r"/[A-Z0-9][\w-]*\.html$", re.IGNORECASE),
    re.compile(r"/products/[\w-]+$", re.IGNORECASE),
    re.compile(r"/product/[\w-]+/?$", re.IGNORECASE),
    re.compile(r"/(?:p|item|dp)/[\w-]+/?$", re.IGNORECASE),
]

_TEXT_TAGS = ("h1", "h2", "h3", "p", "li", "span", "div")
_TOKEN_RE = re.compile(r"[A-Za-zА-Яа-я0-9]+(?:[-_/][A-Za-zА-Яа-я0-9]+)*")


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def _tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(text)


def _token_count(text: str) -> int:
    return len(_tokenize(text))


def _trim_to_tokens(text: str, max_tokens: int) -> str:
    if max_tokens <= 0:
        return ""
    toks = _tokenize(text)
    return " ".join(toks[:max_tokens])


def _is_price(text: str) -> bool:
    text = text.strip()
    return bool(
        re.match(
            r"^[\$€£¥₹]?\s?\d[\d,.\s]*(?:[\$€£¥₹]|usd|eur|byn|rub|pln)?$",
            text,
            re.I,
        )
    )


def _is_noise(text: str) -> bool:
    text = _clean(text)
    if not text:
        return True
    if len(text) < 3:
        return True
    if _BAD_TEXT_RE.search(text):
        return True
    if _is_price(text):
        return True
    if re.fullmatch(r"[\W_]+", text):
        return True
    if sum(ch.isalpha() for ch in text) < 2:
        return True
    return False


def _in_skip_zone(tag: Tag) -> bool:
    for parent in [tag, *tag.parents]:
        if not isinstance(parent, Tag):
            continue
        if parent.name in _SKIP_TAGS:
            return True
        cls = " ".join(parent.get("class", []))
        id_ = parent.get("id", "")
        role = parent.get("role", "")
        check = f"{cls} {id_} {role}"
        if _SKIP_ATTRS.search(check):
            return True
    return False


def _fetch(url: str) -> tuple[Optional[BeautifulSoup], str, int]:
    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser"), response.url, response.status_code
        return None, response.url, response.status_code
    except Exception:
        return None, url, 0


def _dedupe_keep_order(items: list[str]) -> list[str]:
    seen = set()
    out = []
    for x in items:
        key = x.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(x)
    return out


def _from_jsonld(soup: BeautifulSoup) -> list[str]:
    products = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        _walk_ld(data, products)
    return products


def _walk_ld(node, out: list[str]):
    if isinstance(node, list):
        for item in node:
            _walk_ld(item, out)
        return

    if not isinstance(node, dict):
        return

    types = node.get("@type", "")
    types = types if isinstance(types, list) else [types]

    if any(t in ("Product", "IndividualProduct") for t in types):
        name = _clean(node.get("name", ""))
        if name and not _is_noise(name):
            out.append(name)
        return

    if "ItemList" in types:
        for elem in node.get("itemListElement", []):
            _walk_ld(elem, out)

    if "@graph" in node:
        _walk_ld(node["@graph"], out)

    for key, val in node.items():
        if key.startswith("@"):
            continue
        if isinstance(val, (dict, list)):
            _walk_ld(val, out)


def _looks_like_product_url(href: str, base_host: str) -> bool:
    parsed = urlparse(href)
    if parsed.netloc and parsed.netloc != base_host:
        return False
    path = parsed.path
    return any(pat.search(path) for pat in _PRODUCT_URL_PATTERNS)


def _name_from_img_link(a_tag: Tag) -> Optional[str]:
    img = a_tag.find("img")
    if not img:
        return None
    for attr in ("title", "alt"):
        val = _clean(img.get(attr, "") or "")
        if val and not _is_noise(val) and not _is_price(val):
            return val
    return None


def _name_from_text_link(a_tag: Tag) -> Optional[str]:
    lines = [_clean(l) for l in a_tag.get_text("\n").split("\n") if _clean(l)]
    for line in lines:
        if not _is_noise(line) and not _is_price(line):
            return line
    return None


def _from_product_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    base_host = urlparse(base_url).netloc
    found = {}

    for a in soup.find_all("a", href=True):
        href = urljoin(base_url, a["href"])
        norm = urlparse(href)._replace(query="", fragment="").geturl()

        if not _looks_like_product_url(href, base_host):
            continue
        if _in_skip_zone(a):
            continue
        if norm in found:
            continue

        name = _name_from_img_link(a) or _name_from_text_link(a)
        if name:
            found[norm] = name

    return list(found.values())


def _name_from_card(card: Tag) -> Optional[str]:
    for level in ("h1", "h2", "h3", "h4", "h5"):
        h = card.find(level)
        if h:
            txt = _clean(h.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt

    for el in card.find_all(True):
        cls = " ".join(el.get("class", []))
        if re.search(r"(product|item|card)[_-]?(name|title)", cls, re.I):
            txt = _clean(el.get_text())
            if txt and not _is_noise(txt) and not _is_price(txt):
                return txt

    a = card.find("a")
    if a:
        return _name_from_text_link(a)

    return None


def _from_html_cards(soup: BeautifulSoup) -> list[str]:
    products = []
    seen = set()

    candidates = [
        tag for tag in soup.find_all(True)
        if _CARD_PATTERN.search(" ".join(tag.get("class", [])))
    ]

    filtered = []
    for c in candidates:
        if not any(c in f.descendants for f in filtered):
            filtered = [f for f in filtered if f not in c.descendants]
            filtered.append(c)

    for card in filtered:
        if _in_skip_zone(card):
            continue
        name = _name_from_card(card)
        if name and name.lower() not in seen:
            seen.add(name.lower())
            products.append(name)

    return products


def _from_title(soup: BeautifulSoup) -> Optional[str]:
    title_tag = soup.find("title")
    if title_tag:
        raw = _clean(title_tag.get_text())
        txt = re.split(r"\s*[|–—]\s*", raw)[0].strip()
        txt = re.sub(r"\s*[-–]\s*(Shop|Store|Buy|Online).*$", "", txt, flags=re.I).strip()
        if txt and not _is_noise(txt):
            return txt
    return None


def _detect_page_type(products: list[str]) -> str:
    if len(products) >= 6:
        return "catalog"
    return "single"


def _collect_noise_candidates(soup: BeautifulSoup, products: list[str], page_type: str) -> list[str]:
    product_keys = {p.casefold() for p in products}
    seen = set()
    out = []

    for tag in soup.find_all(_TEXT_TAGS):
        if _in_skip_zone(tag):
            continue

        # не берём большие контейнеры поверх уже вложенных текстовых элементов
        if tag.find(_TEXT_TAGS):
            continue

        txt = _clean(tag.get_text(" ", strip=True))
        if _is_noise(txt):
            continue
        if txt.casefold() in product_keys:
            continue

        n = _token_count(txt)

        # режем шум агрессивно
        max_piece = 12 if page_type == "single" else 16
        if n > max_piece:
            txt = _trim_to_tokens(txt, max_piece)
            n = _token_count(txt)

        if n < 2:
            continue

        key = txt.casefold()
        if key in seen:
            continue
        seen.add(key)
        out.append(txt)

    return out


def _take_noise_by_budget(candidates: list[str], min_tokens: int, max_tokens: int) -> list[str]:
    picked = []
    cur = 0

    for txt in candidates:
        if cur >= max_tokens:
            break

        n = _token_count(txt)
        if n <= 0:
            continue

        if cur + n <= max_tokens:
            picked.append(txt)
            cur += n
        else:
            remain = max_tokens - cur
            clipped = _trim_to_tokens(txt, remain)
            if clipped:
                picked.append(clipped)
                cur += _token_count(clipped)
            break

    return picked


def scrape_some_page_info(
    url: str,
    include_status: bool = False,
    shuffle_blocks: bool = True,
    single_noise_ratio: tuple[float, float] = (0.3, 0.6),
    catalog_noise_ratio: tuple[float, float] = (0.5, 0.8),
) -> list[str]:
    soup, final_url, status_code = _fetch(url)
    if not soup:
        return [f"STATUS_{status_code}"] if include_status else []

    products = _dedupe_keep_order(
        _from_jsonld(soup) +
        _from_html_cards(soup) +
        _from_product_links(soup, url)
    )

    title = _from_title(soup)
    if title and title.casefold() not in {p.casefold() for p in products}:
        products.append(title)

    if not products:
        return []

    page_type = _detect_page_type(products)
    product_text = " ".join(products)
    product_tokens = _token_count(product_text)

    if page_type == "single":
        min_noise = ceil(product_tokens * single_noise_ratio[0])
        max_noise = ceil(product_tokens * single_noise_ratio[1])
    else:
        min_noise = ceil(product_tokens * catalog_noise_ratio[0])
        max_noise = ceil(product_tokens * catalog_noise_ratio[1])

    noise_candidates = _collect_noise_candidates(soup, products, page_type)
    noise = _take_noise_by_budget(noise_candidates, min_noise, max_noise)

    blocks = []

    if include_status:
        blocks.append(f"STATUS_{status_code}")

    # важный момент: не держим фиксированный порядок product -> noise
    product_blocks = [f"PRODUCT_BLOCK: {p}" for p in products]
    noise_blocks = [f"NOISE_BLOCK: {n}" for n in noise]

    blocks.extend(product_blocks + noise_blocks)

    if shuffle_blocks:
        rnd = random.Random(hash(url) & 0xffffffff)
        rnd.shuffle(blocks)

    return _dedupe_keep_order([b for b in blocks if b])

In [9]:
print(scrape_some_page_info(X[0]))

['NOISE_BLOCK: FAST SHIPPING AUSTRALIA', 'PRODUCT_BLOCK: Factory Buys 32cm Euro Top Mattress - King', 'PRODUCT_BLOCK: Skip to content', 'NOISE_BLOCK: BED FRAMES FROM $89!', 'PRODUCT_BLOCK: Buy King Factory Buys 32cm Euro Top Mattress', 'NOISE_BLOCK: FREE SHIPPING ON MATTRESSES']


In [10]:
X_train_texts = [scrape_some_page_info(url) for url in X_train]
X_train_texts = [str(text) if text is not None else "" for text in X_train_texts]

In [11]:
from huggingface_hub import login
login()

In [12]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased") 

inputs = tokenizer(X_train_texts, padding = True, truncation = True, 
                   max_length = 512, return_offsets_mapping = True, return_tensors = "pt")

In [13]:
def find_target_positions(text, product):
    if not text:
        return []
    matches = re.finditer(rf'\b{re.escape(product)}\b', text)
    return [(m.start(), m.end()) for m in matches]

In [14]:
offset_mapping = inputs.pop("offset_mapping").cpu().numpy()

all_labels = []

for i in range(len(X_train_texts)):
    text = X_train_texts[i]
    target = y_train.iloc[i] 
    
    target_positions = find_target_positions(text, target)
    
    doc_offsets = offset_mapping[i]
    doc_labels = []
    
    for start, end in doc_offsets:
        if start == 0 and end == 0:
            doc_labels.append(-100)
            continue
        
        is_target = False
        for target_start, target_end in target_positions:
            if start >= target_start and end <= target_end:
                is_target = True
                break
        
        doc_labels.append(1 if is_target else 0)
    
    all_labels.append(doc_labels)

inputs["labels"] = torch.tensor(all_labels)

In [15]:
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = {key: torch.tensor(val) if not torch.is_tensor(val) else val 
                          for key, val in encodings.items()}
        
    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}
    
    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = NERDataset(inputs)

In [16]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [17]:
from transformers import BertForTokenClassification

model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels = 2)

device = torch.device("cuda")
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [18]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size = 4, shuffle = True)

optimizer = AdamW(model.parameters(), lr = 5e-5)
device = torch.device("cuda")
model.to(device)
model.train()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [19]:
from tqdm.auto import tqdm

epochs = 5

for epoch in range(epochs):
    loop = tqdm(train_loader, leave=True)
    epoch_loss = 0
    
    for batch in loop:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss 
        loss.backward()
        
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_description(f'Epoch {epoch+1}')
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} finished. Average Loss: {epoch_loss/len(train_loader):.4f}")

model.save_pretrained(r'C:\dev\projects\NLP_test\res\final_model')

  0%|          | 0/16 [00:00<?, ?it/s]

Epoch 1 finished. Average Loss: 0.1585


  0%|          | 0/16 [00:00<?, ?it/s]

Epoch 2 finished. Average Loss: 0.0949


  0%|          | 0/16 [00:00<?, ?it/s]

...

  0%|          | 0/16 [00:00<?, ?it/s]

Epoch 4 finished. Average Loss: 0.0275


  0%|          | 0/16 [00:00<?, ?it/s]

Epoch 5 finished. Average Loss: 0.0169


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
id2label = {0: "NO_PRODUCT", 1: "PRODUCT"}
label2id = {"NO_PRODUCT": 0, "PRODUCT": 1}

model.config.id2label = id2label
model.config.label2id = label2id

model.save_pretrained(r'C:\dev\projects\NLP_test\res\final_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
def evaluate_model(model, data_loader, dataset_name="Target Set"):
    model.eval() 
    device = torch.device("cuda")
    model.to(device)

    total_loss = 0
    all_predictions = []
    all_true_labels = []

    print(f"Evaluating on {dataset_name}...")

    with torch.no_grad(): 
        for batch in tqdm(data_loader):
            labels = batch['labels'].to(device) 
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            
            loss = outputs.loss
            logits = outputs.logits 

            predictions = torch.argmax(logits, dim=-1)
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    print(f"{dataset_name} Loss: {avg_loss:.4f}")
    
    return avg_loss, all_true_labels, all_predictions

In [22]:
from sklearn.metrics import classification_report

def get_classification_report(model, data_loader, dataset_name="Target Set"):

    label_list = ["NO_PRODUCT", "PRODUCT"]
    model.eval()
    device = torch.device("cuda")
    model.to(device)

    flat_predictions = []
    flat_true_labels = []

    print(f"Calculating metrics on {dataset_name}...")

    with torch.no_grad():
        for batch in tqdm(data_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'] 

            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

            for i in range(len(labels)):
                for j in range(len(labels[i])):
                    label_id = labels[i][j].item()
                    
                    if label_id != -100:
                        flat_true_labels.append(label_list[label_id])
                        flat_predictions.append(label_list[predictions[i][j]])

    print(f"\n{dataset_name} Classification Report")
    report = classification_report(flat_true_labels, flat_predictions, zero_division = 0)
    print(report)

In [23]:
train_loss, train_true, train_pred = evaluate_model(model, train_loader, "Train")

Evaluating on Train...


  0%|          | 0/16 [00:00<?, ?it/s]

Train Loss: 0.0095


In [24]:
get_classification_report(model, train_loader, "Train")

Calculating metrics on Train...


  0%|          | 0/16 [00:00<?, ?it/s]


Train Classification Report
              precision    recall  f1-score   support

  NO_PRODUCT       1.00      1.00      1.00      7697
     PRODUCT       0.98      0.91      0.94       142

    accuracy                           1.00      7839
   macro avg       0.99      0.95      0.97      7839
weighted avg       1.00      1.00      1.00      7839



In [25]:
#test
X_test_texts = [scrape_some_page_info(url) for url in X_test]
X_test_texts = [str(text) if text is not None else "" for text in X_test_texts]

In [26]:
inputs_test = tokenizer(X_test_texts, padding = True, truncation = True, 
                   max_length = 512, return_offsets_mapping = True, return_tensors = "pt")

offset_mapping_test = inputs_test.pop("offset_mapping").cpu().numpy()
all_labels_test = []

for i, text in enumerate(X_test_texts):
    offsets = offset_mapping_test[i]
    target_positions = find_target_positions(text, y_test.iloc[i]) 
    
    doc_labels = []
    for start, end in offsets:
        if start == 0 and end == 0:
            doc_labels.append(-100)
            continue
        is_target = False
        for t_start, t_end in target_positions:
            if start >= t_start and end <= t_end:
                is_target = True
                break
        doc_labels.append(1 if is_target else 0)
    all_labels_test.append(doc_labels)

inputs_test["labels"] = torch.tensor(all_labels_test)

In [27]:
test_dataset = NERDataset(inputs_test)
test_loader = DataLoader(test_dataset, batch_size = 4, shuffle = False)

In [28]:
test_loss, test_true, test_pred = evaluate_model(model, test_loader, "Test")

Evaluating on Test...


  0%|          | 0/4 [00:00<?, ?it/s]

Test Loss: 0.1949


In [29]:
get_classification_report(model, test_loader, "Test")

Calculating metrics on Test...


  0%|          | 0/4 [00:00<?, ?it/s]


Test Classification Report
              precision    recall  f1-score   support

  NO_PRODUCT       0.97      0.99      0.98      1625
     PRODUCT       0.58      0.30      0.40        60

    accuracy                           0.97      1685
   macro avg       0.78      0.65      0.69      1685
weighted avg       0.96      0.97      0.96      1685

